In [ ]:
import http.client
import pandas as pd
import json
import os
import numpy as np
import time
import requests
from datetime import datetime, timedelta
import re, sys
from dotenv import load_dotenv
from pathlib import Path

## Scraping Weather information

In [ ]:
#chemin vers le fichier .env
dotenv_path = Path("/Users/pierre-yvesbenevise/Documents/DataEngineer/Projects/.env")

# charger le fichier .env
load_dotenv(dotenv_path=dotenv_path)
rapidapi_key = os.getenv("RAPIDAPI_KEY")

In [ ]:
conn = http.client.HTTPSConnection("meteostat.p.rapidapi.com")

headers = {
    'x-rapidapi-key': rapidapi_key,
    'x-rapidapi-host': "meteostat.p.rapidapi.com"
}

### Finding NYC weather station id

In [41]:
#to find NYC id station

conn.request("GET", "/stations/meta?icao=KNYC", headers=headers)

res = conn.getresponse()

station = json.loads(res.read().decode("utf-8"))['data']['id']
print(station)

KNYC0


### Create urls list between two dates

In [42]:
# Station and timezone
tz = "Europe/Berlin"

# Start and end date

start_date = datetime(2024, 11, 1)
end_date = datetime(2025, 12, 1)

urls_to_call = []

current_start = start_date
while current_start < end_date:
    current_end = current_start + timedelta(days=29)  # 30-day interval
    if current_end > end_date:
        current_end = end_date

    url = f"/stations/hourly?station={station}&start={current_start.date()}&end={current_end.date()}&tz={tz}"
    urls_to_call.append(url)

    current_start = current_end + timedelta(days=1)  # next interval

print("Number of requests:", len(urls_to_call))

Number of requests: 14


### Sending requests, building dataframe, saving as json

In [43]:
weather_df = pd.DataFrame()
all_weather_entries = []

MAX_RPS = 3
MIN_INTERVAL = 1 / MAX_RPS  # 0.333 seconds -> max allowed with Free acount
last_request_time = 0

print('start request...')
for url in urls_to_call:
    now = time.time()
    elapsed = now - last_request_time

    if elapsed < MIN_INTERVAL:
        time.sleep(MIN_INTERVAL - elapsed)

    conn.request("GET", url, headers=headers)
    res = conn.getresponse()

    weather_data = json.loads(res.read().decode("utf-8"))

    all_weather_entries.extend(weather_data["data"])

    print("request from {} to {}".format(url.split("start=")[1][:10], url.split("end=")[1][:10]))

    last_request_time = time.time()

print('...end request')

#create DataFrame
weather_df = pd.DataFrame(all_weather_entries)
print('weather data loaded into dataframe')

# Convert time
weather_df["time"] = pd.to_datetime(weather_df["time"])

# Rename columns
weather_df = weather_df.rename(columns={
    "rhum": "relative_humidity",
    "prcp": "precipitation_total",
    "snow": "snow_depth",
    "wspd": "average_wind_speed",
    "tsun": "sunshine_total",
})
print('renaming columns done')

#remove useless columns
weather_df =  weather_df.drop(['dwpt', 'wdir', 'wpgt', 'pres'], axis = 1)
print('dropping dwpt, wdir, wpgt, pres columns')



start request...
request from 2024-11-01 to 2024-11-30
request from 2024-12-01 to 2024-12-30
request from 2024-12-31 to 2025-01-29
request from 2025-01-30 to 2025-02-28
request from 2025-03-01 to 2025-03-30
request from 2025-03-31 to 2025-04-29
request from 2025-04-30 to 2025-05-29
request from 2025-05-30 to 2025-06-28
request from 2025-06-29 to 2025-07-28
request from 2025-07-29 to 2025-08-27
request from 2025-08-28 to 2025-09-26
request from 2025-09-27 to 2025-10-26
request from 2025-10-27 to 2025-11-25
request from 2025-11-26 to 2025-12-01
...end request
weather data loaded into dataframe
renaming columns done
dropping dwpt, wdir, wpgt, pres columns


In [44]:
weather_df.info()
display(weather_df.head())
display(weather_df.tail())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9504 entries, 0 to 9503
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   time                 9504 non-null   datetime64[ns]
 1   temp                 9310 non-null   float64       
 2   relative_humidity    9310 non-null   float64       
 3   precipitation_total  9258 non-null   float64       
 4   snow_depth           0 non-null      object        
 5   average_wind_speed   9307 non-null   float64       
 6   sunshine_total       0 non-null      object        
 7   coco                 9258 non-null   float64       
dtypes: datetime64[ns](1), float64(5), object(2)
memory usage: 594.1+ KB


,time,temp,relative_humidity,precipitation_total,snow_depth,average_wind_speed,sunshine_total,coco
0,2024-11-01 00:00:00,23.0,52.0,0.0,None,7.0,None,3.0
1,2024-11-01 01:00:00,24.0,50.0,0.0,None,15.0,None,3.0
2,2024-11-01 02:00:00,23.0,52.0,0.0,None,7.0,None,3.0
3,2024-11-01 03:00:00,23.0,55.0,0.0,None,22.7,None,3.0
4,2024-11-01 04:00:00,22.0,55.0,0.0,None,11.0,None,3.0


,time,temp,relative_humidity,precipitation_total,snow_depth,average_wind_speed,sunshine_total,coco
9499,2025-12-01 19:00:00,6.0,29.0,0.0,None,6.0,None,1.0
9500,2025-12-01 20:00:00,5.0,29.0,0.0,None,11.0,None,3.0
9501,2025-12-01 21:00:00,4.0,31.0,0.0,None,9.0,None,2.0
9502,2025-12-01 22:00:00,4.0,33.0,0.0,None,6.0,None,3.0
9503,2025-12-01 23:00:00,4.0,34.0,0.0,None,7.2,None,3.0


In [45]:
#Pourcentage de vide
print('Pourcentage de vide')
missing_pct = 100 * weather_df.isnull().sum() / weather_df.shape[0]
missing_pct = missing_pct[missing_pct > 0].sort_values(ascending=False)
print(missing_pct)

Pourcentage de vide
snow_depth             100.000000
sunshine_total         100.000000
precipitation_total      2.588384
coco                     2.588384
average_wind_speed       2.072811
temp                     2.041246
relative_humidity        2.041246
dtype: float64


In [46]:
#enlève collonnes vides: snwo_depth, and sunshine_total
weather_df =  weather_df.drop(['snow_depth', 'sunshine_total'], axis = 1)
print('supprimer colone car vide snow_depth, wdir, sunshine_total')

supprimer colone car vide snow_depth, wdir, sunshine_total


In [47]:
dups = weather_df[weather_df.duplicated(subset=["time"], keep=False)]
print('Nbre de doublons sur la colonne time:', dups.shape[0])

Nbre de doublons sur la colonne time: 2


-> Il y a 4 time en doublon

In [48]:
dups.sort_values("time")

,time,temp,relative_humidity,precipitation_total,average_wind_speed,coco
8617,2025-10-26 02:00:00,13.0,45.0,0.0,0.0,3.0
8618,2025-10-26 02:00:00,12.0,49.0,0.0,6.0,3.0


-> supprimer les doublons et ne garder que la moyenne des valeurs sur les autres colones

In [49]:
weather_df = weather_df.groupby("time", as_index=False).mean(numeric_only=True)
dups = weather_df[weather_df.duplicated(subset=["time"], keep=False)]
print('Nbre de doublons sur la colonne time:', dups.shape[0])

Nbre de doublons sur la colonne time: 0


In [50]:
print("Nbre de lignes avec au moins une valeur NaN:", weather_df.isna().any(axis=1).sum())
nan_rows = weather_df[weather_df.isna().any(axis=1)]
print(nan_rows)


Nbre de lignes avec au moins une valeur NaN: 246
                    time  temp  relative_humidity  precipitation_total  \
938  2024-12-10 02:00:00   NaN                NaN                  NaN   
939  2024-12-10 03:00:00   NaN                NaN                  NaN   
940  2024-12-10 04:00:00   NaN                NaN                  NaN   
941  2024-12-10 05:00:00   NaN                NaN                  NaN   
942  2024-12-10 06:00:00   NaN                NaN                  NaN   
...                  ...   ...                ...                  ...   
3374 2025-03-21 14:00:00   4.0               44.0                  NaN   
3375 2025-03-21 15:00:00   5.0               38.0                  NaN   
3376 2025-03-21 16:00:00   6.0               33.0                  NaN   
3377 2025-03-21 17:00:00   8.0               27.0                  NaN   
3378 2025-03-21 18:00:00   8.0               25.0                  NaN   

      average_wind_speed  coco  
938                  NaN   Na

In [51]:
#drop na 
print("supprimer les lignes avec au moins une valeur NaN")
weather_df = weather_df.dropna()
#Pourcentage de vide
print('Pourcentage de vide')
missing_pct = 100 * weather_df.isnull().sum() / weather_df.shape[0]
missing_pct = missing_pct[missing_pct > 0].sort_values(ascending=False)
print(missing_pct)

supprimer les lignes avec au moins une valeur NaN
Pourcentage de vide
Series([], dtype: float64)


In [52]:
#créer une timeline horaire complète
full_time = pd.date_range(
    start=weather_df['time'].min(),
    end=weather_df['time'].max(),
    freq='h'
)

#Mettre à jour le DataFrame pour inclure toutes les heures, en remplissant 
# les valeurs manquantes par la dernière observation connue (forward fill)
weather_df = (
    weather_df
    .set_index('time')
    .reindex(full_time)
    .ffill()
    .reset_index()
    .rename(columns={'index': 'time'})
)

In [53]:
#extrait la date and l'heure pour faire la jointure future
weather_df['date'] = pd.to_datetime(weather_df['time'].dt.date)
weather_df['hour'] = weather_df['time'].dt.hour

# Sanity check
dups = weather_df[weather_df.duplicated(subset=["date", 'hour'], keep=False)]
print('Doublon (date, heure)', dups.shape[0])

#supprimer la colonne time
weather_df =  weather_df.drop(['time'], axis = 1)
print('supprimer colonne time')
weather_df.info()

Doublon (date, heure) 0
supprimer colonne time
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9504 entries, 0 to 9503
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   temp                 9504 non-null   float64       
 1   relative_humidity    9504 non-null   float64       
 2   precipitation_total  9504 non-null   float64       
 3   average_wind_speed   9504 non-null   float64       
 4   coco                 9504 non-null   float64       
 5   date                 9504 non-null   datetime64[ns]
 6   hour                 9504 non-null   int32         
dtypes: datetime64[ns](1), float64(5), int32(1)
memory usage: 482.8 KB


In [2]:
weather_df = weather_df.astype({
    "date": "datetime64[ns]",
    'hour': 'int32',
    "temp": "float32",
    "relative_humidity": "float32",
    "precipitation_total": "float32",
    "average_wind_speed": "float32",
    "coco": "int32"
})
print('changing type columns')
weather_df.info()

NameError: name 'weather_df' is not defined

In [55]:
#réeordonner les colonnes
weather_df = weather_df[
    [
        'date',
        'hour',
        'temp',
        'relative_humidity',
        'precipitation_total',
        'average_wind_speed',
        'coco'
    ]
]
weather_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9504 entries, 0 to 9503
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   date                 9504 non-null   datetime64[ns]
 1   hour                 9504 non-null   int32         
 2   temp                 9504 non-null   float32       
 3   relative_humidity    9504 non-null   float32       
 4   precipitation_total  9504 non-null   float32       
 5   average_wind_speed   9504 non-null   float32       
 6   coco                 9504 non-null   int32         
dtypes: datetime64[ns](1), float32(4), int32(2)
memory usage: 297.1 KB


In [56]:
# export file to csv format
filename = "./data/weather_data" + "_" + start_date.strftime("%Y_%m_%d") + "_" + end_date.strftime("%Y_%m_%d") + ".csv"

#remove all files of the same start and end dates
if os.path.exists(filename):
    os.remove(filename)

# save data
weather_df.to_csv(filename, index=False, encoding="utf-8")

print('file saved', filename)

file saved ./data/weather_data_2024_11_01_2025_12_01.csv


## Scraping Holidays information

In [65]:
out_date_format = "%Y-%m-%d"

if len(sys.argv) == 3:
    out_file, out_date_format = sys.argv[1:]

rows = []

r = requests.get("https://www.opm.gov/policy-data-oversight/pay-leave/federal-holidays/#url=2024")
r.raise_for_status()

html = r.text.split('<section class="tab-content" title="2024">')[1]

for table in re.findall(r'<table class="DataTable HolidayTable">([\s\S]*?)</table>', html):
    year = re.findall(r"(\d{4}) Holiday Schedule", table)
    if len(year) == 1:
        year = year[0]
    else:
        continue

    for tr in re.findall(r"<tr>([\s\S]*?)</tr>", table):
        row = []
        for td in re.findall(r"<td>([\s\S]*?)</td>", tr):
            row.append(re.sub(r"<.*>", "", td).replace("’", "'").strip())
        if len(row) == 2:
            years = re.findall("\d{4}", row[0])
            if years and years[0] != year:
                continue # this table specified for a different year
            date = row[0].split(", ")[1] + ", " + year
            date = time.strptime(date, "%B %d, %Y")
            row[0] = time.strftime(out_date_format, date)
            rows.append(row)

holidays_df = pd.DataFrame(rows, columns=["date", "holiday"])
holidays_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 211 entries, 0 to 210
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   date     211 non-null    object
 1   holiday  211 non-null    object
dtypes: object(2)
memory usage: 3.4+ KB


In [66]:
#transform string to datatime format
holidays_df['date'] = pd.to_datetime(holidays_df['date'].apply(lambda x : datetime.strptime(x, "%Y-%m-%d").date()))

# Add a boolean column to mark if it is a holiday
holidays_df['is_holiday'] = holidays_df['holiday'].notna().astype(bool)

#Supprimer la colonne holiday
holidays_df = holidays_df.drop(columns=['holiday'], axis=1)

holidays_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 211 entries, 0 to 210
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   date        211 non-null    datetime64[ns]
 1   is_holiday  211 non-null    bool          
dtypes: bool(1), datetime64[ns](1)
memory usage: 2.0 KB


In [69]:
#Garder les dates uniquement compris dans les dates de l'étude
start_date = weather_df['date'].min().normalize()
end_date = weather_df['date'].max().normalize()

holidays_df = holidays_df[holidays_df['date'].between(start_date, end_date)]
holidays_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 14 entries, 8 to 21
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   date        14 non-null     datetime64[ns]
 1   is_holiday  14 non-null     bool          
dtypes: bool(1), datetime64[ns](1)
memory usage: 238.0 bytes


In [70]:
# Sanity check
dups = holidays_df[holidays_df.duplicated(subset=["date"], keep=False)]
print('Import holidays data  duplicate', dups.shape[0])
print(dups.head())
print("Dropping dupplicate holidays")
holidays_df = holidays_df.drop_duplicates(subset=["date"])
dups = holidays_df[holidays_df.duplicated(subset=["date"], keep=False)]
print('Import holidays data duplicate', dups.shape[0])

Import holidays data  duplicate 2
         date  is_holiday
12 2025-01-20        True
13 2025-01-20        True
Dropping dupplicate holidays
Import holidays data duplicate 0


In [71]:
# export file to csv format
filename = "./data/holidays.csv"

#remove all files of the same start and end dates
if os.path.exists(filename):
    os.remove(filename)

# save data
holidays_df.to_csv(filename, index=False, encoding="utf-8")

print('file saved', filename)

file saved ./data/holidays.csv
